# NB4 — Training / T-22 protocol. **Kaggle v5e-8**.

Переписан по протоколу `tools/t0_diagnose.py`: все branches (base/bs2_32/lean/highest) стартуют из ОДНОЙ точки (тот же seed, та же инициализация), чтобы расхождение доказывало эффект настройки, а не разницу инициализации. Resume из checkpoint поддержан. Использует `shard_map` (Tier 0 фикс #2) — раньше на v5e-8 использовался только 1 из 8 чипов.

In [1]:
# =============================================================================
# CELL 0 — robust platform detection + clean
# =============================================================================
%env NB_ID=nb4
import os, sys

# ---------- reliable platform detection ----------
def _detect_platform():
    """Порядок: env vars → FS markers → import google.colab → cwd.
    Возвращает 'colab', 'kaggle' или 'unknown'."""
    # 1. env vars — самые надёжные, платформа ставит их до Python
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_TPU_ADDR"):
        return "colab"
    # 2. FS markers, но только один должен быть
    has_kaggle = os.path.isdir("/kaggle/working")
    has_colab  = os.path.isdir("/content")
    if has_kaggle and not has_colab:
        return "kaggle"
    if has_colab and not has_kaggle:
        return "colab"
    # 3. import google.colab — есть только в Colab
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        pass
    # 4. cwd fallback
    cwd = os.getcwd()
    if cwd.startswith("/kaggle"):
        return "kaggle"
    if cwd.startswith("/content"):
        return "colab"
    # 5. оба маркера есть → приоритет Kaggle (/content там артефакт)
    if has_kaggle and has_colab:
        return "kaggle"
    return "unknown"

PLATFORM = _detect_platform()
IS_COLAB  = (PLATFORM == "colab")
IS_KAGGLE = (PLATFORM == "kaggle")
assert IS_COLAB or IS_KAGGLE, (
    f"Не удалось определить платформу. cwd={os.getcwd()} "
    f"has_content={os.path.isdir('/content')} "
    f"has_kaggle={os.path.isdir('/kaggle/working')} "
    f"KAGGLE_KERNEL_RUN_TYPE={os.environ.get('KAGGLE_KERNEL_RUN_TYPE')} "
    f"COLAB_RELEASE_TAG={os.environ.get('COLAB_RELEASE_TAG')}"
)

EXPECTED_PLATFORM = "kaggle"
if PLATFORM != EXPECTED_PLATFORM:
    print(f"[WARN] ноутбук предназначен для {EXPECTED_PLATFORM}, запущен на {PLATFORM}")
    print(f"[WARN] NB_ID={os.environ.get('NB_ID')} — продолжай на свой риск.")
else:
    print(f"[PLATFORM] {PLATFORM} ✓")

WORK_ROOT = "/content" if IS_COLAB else "/kaggle/working"
print(f"[ENV] cwd={os.getcwd()}  has /content={os.path.isdir('/content')}  "
      f"has /kaggle/working={os.path.isdir('/kaggle/working')}")
print(f"[ENV] KAGGLE_KERNEL_RUN_TYPE={os.environ.get('KAGGLE_KERNEL_RUN_TYPE')}  "
      f"COLAB_RELEASE_TAG={os.environ.get('COLAB_RELEASE_TAG')}")
print(f"python={sys.version.split()[0]}  root={WORK_ROOT}  NB_ID={os.environ.get('NB_ID')}")

for m in list(sys.modules.keys()):
    if (m in ("jax","jaxlib","libtpu","flax","optax","orbax","atomic_ops")
        or m.startswith(("jax.","jaxlib.","flax.","optax.","orbax.","atomic_ops."))):
        sys.modules.pop(m, None)
import importlib; importlib.invalidate_caches()
print("[clean] system jax dropped from sys.modules")

env: NB_ID=nb4
[PLATFORM] kaggle ✓
[ENV] cwd=/kaggle/working  has /content=False  has /kaggle/working=True
[ENV] KAGGLE_KERNEL_RUN_TYPE=Interactive  COLAB_RELEASE_TAG=None
python=3.12.13  root=/kaggle/working  NB_ID=nb4
[clean] system jax dropped from sys.modules


In [2]:
# =============================================================================
# CELL 1 — venv setup (идемпотентно)
# =============================================================================
import subprocess, os, sys, shutil, glob

VENV_DIR  = f"{WORK_ROOT}/venv"
VENV_PY   = f"{VENV_DIR}/bin/python"
VENV_PIP  = f"{VENV_DIR}/bin/pip"
_PYTAG    = f"python{sys.version_info.major}.{sys.version_info.minor}"
VENV_SITE = f"{VENV_DIR}/lib/{_PYTAG}/site-packages"

print("=" * 78); print("CELL 1 — SETUP VENV"); print("=" * 78)
print(f"[paths] VENV_DIR={VENV_DIR}")

venv_ok = (os.path.isfile(VENV_PY) and os.path.isfile(VENV_PIP)
           and os.path.isdir(VENV_SITE))

if venv_ok:
    r = subprocess.run([VENV_PY, "-c",
        "import jax, jaxlib; print(f'jax={jax.__version__} jaxlib={jaxlib.__version__}')"],
        capture_output=True, text=True, timeout=60)
    if r.returncode == 0:
        print(f"[skip] venv OK: {r.stdout.strip()}")
    else:
        print("[warn] venv сломан, пересоздаю")
        shutil.rmtree(VENV_DIR, ignore_errors=True); venv_ok = False

if not venv_ok:
    print("[1] создаю venv ...")
    try:
        import virtualenv  # noqa
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--user",
                        "virtualenv"], capture_output=True, text=True, timeout=300)
    r = subprocess.run([sys.executable, "-m", "virtualenv",
                        "--python", sys.executable, VENV_DIR],
                       capture_output=True, text=True, timeout=600)
    if r.returncode != 0 or not os.path.isfile(VENV_PIP):
        shutil.rmtree(VENV_DIR, ignore_errors=True)
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", VENV_DIR],
                       capture_output=True, text=True, timeout=300)
        import urllib.request
        getpip = "/tmp/get-pip.py"
        urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", getpip)
        subprocess.run([VENV_PY, getpip], capture_output=True, text=True, timeout=300)

    subprocess.run([VENV_PIP, "install", "-q", "--upgrade",
                    "pip", "setuptools", "wheel"],
                   capture_output=True, text=True, timeout=300)

    print("[2] установка jax[tpu]==0.11.2 + flax==0.12.9 + optax + atomic-ops ...")
    r = subprocess.run([
        VENV_PIP, "install", "--force-reinstall", "--no-cache-dir",
        "jax[tpu]==0.11.2", "flax==0.12.9", "optax>=0.2.0",
        "orbax-checkpoint", "atomic-ops", "kagglehub",
        "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
    ], capture_output=True, text=True, timeout=2400)
    print(f"    rc={r.returncode}")
    if r.returncode != 0:
        print("STDOUT:", r.stdout[-1500:]); print("STDERR:", r.stderr[-1500:])
        raise RuntimeError("pip install failed")
    print("[3] venv готов")

print("\n[verify] ключевые пакеты:")
for pkg in ("jax","jaxlib","libtpu","flax","optax","atomic_ops"):
    dirs = (glob.glob(f"{VENV_SITE}/{pkg}-*.dist-info")
            + glob.glob(f"{VENV_SITE}/{pkg.replace('_','-')}-*.dist-info"))
    ver = "?"
    if dirs and os.path.isfile(f"{dirs[0]}/METADATA"):
        with open(f"{dirs[0]}/METADATA") as f:
            for line in f:
                if line.startswith("Version:"):
                    ver = line.split(":", 1)[1].strip(); break
    exists = (os.path.isdir(f"{VENV_SITE}/{pkg}") or
              os.path.isfile(f"{VENV_SITE}/{pkg}/__init__.py"))
    print(f"  [{'OK ' if exists else 'MISS'}] {pkg}={ver}")

CELL 1 — SETUP VENV
[paths] VENV_DIR=/kaggle/working/venv
[1] создаю venv ...
[2] установка jax[tpu]==0.11.2 + flax==0.12.9 + optax + atomic-ops ...
    rc=0
[3] venv готов

[verify] ключевые пакеты:
  [OK ] jax=0.11.2
  [OK ] jaxlib=0.11.2
  [OK ] libtpu=0.0.48
  [OK ] flax=0.12.9
  [OK ] optax=0.2.8
  [OK ] atomic_ops=0.1.0


In [3]:
# =============================================================================
# CELL 2 — activate venv + TPU init
# =============================================================================
import os, sys, time, json, math, gc, importlib, hashlib, glob, subprocess
import numpy as np

print("=" * 78); print("CELL 2 — ACTIVATION"); print("=" * 78)
assert os.path.isdir(VENV_SITE), f"venv не найден. Запусти Cell 1."

_killed = 0
for m in list(sys.modules.keys()):
    if (m in ("jax","jaxlib","libtpu","flax","optax","orbax","atomic_ops")
        or m.startswith(("jax.","jaxlib.","flax.","optax.","orbax.","atomic_ops."))):
        sys.modules.pop(m, None); _killed += 1
importlib.invalidate_caches()
print(f"[clean] removed {_killed} modules")

sys.path = [p for p in sys.path if p != VENV_SITE]
sys.path.insert(0, VENV_SITE)

import jax, jaxlib
print(f"[jax] {jax.__version__} from {jax.__file__}")
assert VENV_DIR in jax.__file__, f"jax не из venv. Restart session."

try:
    import libtpu; print(f"[jax] libtpu={libtpu.__version__}")
except Exception as e:
    print(f"[jax] libtpu NA: {e}")

def _try_backend(max_wait=30):
    for i in range(max_wait):
        try:
            return jax.default_backend(), jax.device_count()
        except RuntimeError as e:
            if any(s in str(e) for s in ("vfio","busy","TPU initialization")):
                if i == 0: print(f"  [retry] vfio busy, wait {max_wait}s ...")
                time.sleep(1.0); continue
            print(f"  [fail] {type(e).__name__}: {str(e)[:300]}")
            return None, None
        except Exception as e:
            print(f"  [fail] {type(e).__name__}: {str(e)[:300]}")
            return None, None
    return None, None

backend, ndev = _try_backend()
if backend is None:
    raise RuntimeError("TPU busy — Runtime → Disconnect and delete")
print(f"[jax] backend={backend} ndev={ndev}")
assert backend == "tpu", f"Требуется TPU. backend={backend}"

import jax.numpy as jnp
import optax
from flax import linen as nn

print("[cell2] activation OK")

CELL 2 — ACTIVATION
[clean] removed 0 modules


/kaggle/working/venv/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


[jax] 0.11.2 from /kaggle/working/venv/lib/python3.12/site-packages/jax/__init__.py
[jax] libtpu=0.0.48


E0000 00:00:1790414168.529751      14 common_lib.cc:957] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:239


[jax] backend=tpu ndev=8
[cell2] activation OK


In [4]:
# =============================================================================
# CELL 3 — clone + hashes + archive
# =============================================================================
REPO_URL = "https://github.com/Akseleu-J/gdn2-test.git"
REPO_DIR = f"{WORK_ROOT}/gdn2-test"
PKG_ROOT = os.path.join(REPO_DIR, "pkg")

if not os.path.isdir(REPO_DIR):
    print(f"[clone] {REPO_URL}")
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                       capture_output=True, text=True, timeout=300)
    print(f"    rc={r.returncode}")

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

FILES = sorted(glob.glob(os.path.join(PKG_ROOT, "**", "*.py"), recursive=True))
FILES_HASH = {os.path.relpath(p, PKG_ROOT): _sha(p) for p in FILES}
COMMIT_HASH = hashlib.sha256(
    json.dumps(FILES_HASH, sort_keys=True).encode()).hexdigest()[:16]

print(f"[BOOT] commit={COMMIT[:12]}  files={len(FILES)}  hash={COMMIT_HASH}")

for m in list(sys.modules.keys()):
    if m == "atomic_gdn2" or m.startswith("atomic_gdn2."):
        sys.modules.pop(m, None)
importlib.invalidate_caches()
from atomic_gdn2 import make_cfg, make_blr_trainable
print("[cell3] atomic_gdn2 imported")

OUT_DIR = f"{WORK_ROOT}/attestation"
os.makedirs(OUT_DIR, exist_ok=True)

ENV_INFO = {"commit": COMMIT, "commit_hash": COMMIT_HASH,
            "jax": jax.__version__, "backend": backend, "ndev": ndev,
            "python": sys.version.split()[0],
            "libtpu": getattr(libtpu, "__version__", "n/a") if "libtpu" in sys.modules else "n/a"}

class Archive:
    def __init__(self, nb_id):
        self.path = os.path.join(OUT_DIR, f"results_{nb_id}.json")
        self.data = {"nb_id": nb_id, "env": ENV_INFO, "cells": {}}
        if os.path.isfile(self.path):
            with open(self.path) as f: self.data = json.load(f)
        self._save()
    def _save(self):
        with open(self.path, "w") as f: json.dump(self.data, f, indent=2, default=str)
    def write(self, cid, payload):
        self.data["cells"][cid] = {"ts": time.time(), "payload": payload}
        self._save(); print(f"  [ARCHIVE] {cid}: {payload.get('status','OK')}")

ARCHIVE = Archive(os.environ.get("NB_ID", "nbX"))
print(f"[BOOT] archive at {ARCHIVE.path}")

def rel_l2(a, b, floor=1e-12):
    a = np.asarray(jax.device_get(a), np.float64)
    b = np.asarray(jax.device_get(b), np.float64)
    num = np.linalg.norm((a - b).ravel()); den = np.linalg.norm(b.ravel())
    return float(num if den < floor else num / den)

def isfin(x):
    return bool(np.all(np.isfinite(np.asarray(jax.device_get(x)))))

def hbm_peak_gb(i=0):
    try: return jax.devices()[i].memory_stats().get("peak_bytes_in_use", 0) / 1e9
    except Exception: return -1.0

def bench_med(fn, *args, n=10, w=3):
    jf = jax.jit(fn)
    for _ in range(w): jax.block_until_ready(jf(*args))
    ts = []
    for _ in range(n):
        t = time.perf_counter()
        jax.block_until_ready(jf(*args)); ts.append((time.perf_counter()-t)*1e3)
    return float(np.median(ts)), float(np.std(ts))

[clone] https://github.com/Akseleu-J/gdn2-test.git
    rc=0
[BOOT] commit=b5d408ac7329  files=33  hash=c6862ced7da70782
[cell3] atomic_gdn2 imported
[BOOT] archive at /kaggle/working/attestation/results_nb4.json


In [13]:
# =============================================================================
# TOLERANCES — единый источник допусков
# =============================================================================
TOL = {
    "kernel_vs_f64":        5e-5,
    "grad_vs_ref":          5e-3,
    "t4_finite_diff":       5e-3,
    "old_baseline_rel_o_base": 5e-2,
    "old_baseline_accum_per_L": 2e-4,
    "prod_baseline":        5e-2,
    "anti_test_min":        1e-2,
    "bit_parity":           1e-9,
    "btl_theory_vs_measured_rel": 0.10,
}

def old_baseline_tol(L):
    return TOL["old_baseline_rel_o_base"] + TOL["old_baseline_accum_per_L"] * L

print("[TOL] tolerances loaded:", {k: v for k, v in TOL.items() if not callable(v)})

[TOL] tolerances loaded: {'kernel_vs_f64': 5e-05, 'grad_vs_ref': 0.005, 't4_finite_diff': 0.005, 'old_baseline_rel_o_base': 0.05, 'old_baseline_accum_per_L': 0.0002, 'prod_baseline': 0.05, 'anti_test_min': 0.01, 'bit_parity': 1e-09, 'btl_theory_vs_measured_rel': 0.1}


In [14]:
# =============================================================================
# DRY-RUN — CPU/interpret smoke test
# =============================================================================
print("=" * 78); print("DRY-RUN (interpret=True, tiny shapes)"); print("=" * 78)
try:
    _dr_cfg = make_cfg(bt=64, bc=32, score_bs=64, bs2=32, mb=8, interpret=True)
    _dr_fn = make_blr_trainable(_dr_cfg, 1.0/8.0)
    _dr_rng = np.random.default_rng(0)
    _dr_sh = (1, 1, 64, 32)
    _dr_args = tuple(jnp.asarray((_dr_rng.normal(size=_dr_sh)*0.1).astype(np.float32))
                      for _ in range(3))
    _dr_w = jnp.ones(_dr_sh, jnp.float32)
    _dr_b = jnp.asarray(np.abs(_dr_rng.normal(size=_dr_sh)*0.5).astype(np.float32))
    _dr_g = -jnp.asarray(np.abs(_dr_rng.normal(size=_dr_sh)*0.05).astype(np.float32))
    _dr_o, _dr_h = _dr_fn(_dr_args[0], _dr_args[1], _dr_args[2], _dr_w, _dr_b, _dr_g)
    assert isfin(_dr_o) and isfin(_dr_h), "dry-run forward produced non-finite output"
    _dr_loss = lambda *a: jnp.sum(_dr_fn(*a)[0]**2)
    _dr_grads = jax.grad(_dr_loss, argnums=(0,1,2,3,4,5))(
        _dr_args[0], _dr_args[1], _dr_args[2], _dr_w, _dr_b, _dr_g)
    assert all(isfin(g) for g in _dr_grads), "dry-run backward produced non-finite grad"
    print("[DRY-RUN] PASS — forward+backward finite on tiny interpret=True shapes")
    ARCHIVE.write("0.dryrun", {"status": "PASS"})
except Exception as e:
    print(f"[DRY-RUN] FAIL: {type(e).__name__}: {e}")
    ARCHIVE.write("0.dryrun", {"status": "FAIL", "reason": f"{type(e).__name__}: {str(e)[:300]}"})
    raise RuntimeError("Dry-run failed — не тратим TPU-время на сломанный API.") from e

DRY-RUN (interpret=True, tiny shapes)
[DRY-RUN] PASS — forward+backward finite on tiny interpret=True shapes
  [ARCHIVE] 0.dryrun: PASS


In [15]:
BLR_CFGS, BLR_HASH = {}, {}
for name, kw in (("A17", dict(bt=128, bc=64, score_bs=128, bs2=128, mb=8)),
                 ("A16", dict(bt=128, bc=64, score_bs=128, bs2=64,  mb=8)),
                 ("A15", dict(bt=128, bc=64, score_bs=128, bs2=32,  mb=8))):
    cfg = make_cfg(**kw)
    try: cfg = cfg.with_(b3_dot_mode="bf16x3", b4_dot_mode="bf16x3", diag="btl")
    except Exception: pass
    BLR_CFGS[name] = cfg
    BLR_HASH[name] = hashlib.md5(json.dumps(kw, sort_keys=True).encode()).hexdigest()[:8]
    print(f"[cfg] {name} hash={BLR_HASH[name]}")
SCALE = 1.0 / math.sqrt(128)

BLR_CFGS["A17_SAFE"] = make_cfg(bt=128, bc=64, score_bs=128, bs2=128, mb=8, g_max=160.0/128)

[cfg] A17 hash=36638f52
[cfg] A16 hash=c8e4b010
[cfg] A15 hash=b2d06e75


In [16]:
# =============================================================================
# SHARDING — mesh + shard_map
# =============================================================================
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
try:
    from jax import shard_map
except ImportError:
    from jax.experimental.shard_map import shard_map

_devs = jax.devices(); N_DEV = len(_devs)
mesh = Mesh(np.array(_devs).reshape(-1), ("fsdp",))
print(f"[shard] N_DEV={N_DEV}  mesh={mesh.axis_names}")

def fsdp_spec(shape, n_dev=N_DEV, axis="fsdp"):
    if n_dev <= 1: return P()
    c = [(s, i) for i, s in enumerate(shape) if s >= n_dev and s % n_dev == 0]
    if not c: return P()
    _, i = max(c)
    spec = [None]*len(shape); spec[i] = axis
    return P(*spec)

def shard_of(x, n_dev=N_DEV):
    return NamedSharding(mesh, fsdp_spec(x.shape, n_dev))

def make_sharded_blr_kernel(cfg, scale, H, D):
    """Оборачивает make_blr_trainable в shard_map по батчу.
    h0 создаётся ВНУТРИ local() — обязательно для Pallas+shard_map."""
    fn = make_blr_trainable(cfg, scale)
    sw = lambda t: jnp.swapaxes(t, 1, 2)
    def local(q, k, v, w, b, g):
        B = q.shape[0]
        h0 = jnp.zeros((B, H, D, D), jnp.float32)
        o, hf = fn(sw(q), sw(k), sw(v), sw(w), sw(b), sw(g), h0)
        return sw(o), hf
    def call(q, k, v, w, b, g):
        B = q.shape[0]
        if N_DEV == 1:
            return local(q, k, v, w, b, g)
        if B % N_DEV == 0:
            spec = P("fsdp", None, None, None)
        else:
            spec = P()
        return shard_map(local, mesh=mesh, in_specs=(spec,)*6,
                          out_specs=(spec, spec), check_vma=False)(q, k, v, w, b, g)
    return call

def sharded_init(model, rng, dummy):
    """model.init через out_shardings, а не напрямую."""
    abstract = jax.eval_shape(lambda r: model.init(r, dummy)["params"], rng)
    out_sh = jax.tree.map(shard_of, abstract)
    for _ in range(3): gc.collect()
    jax.clear_caches()
    _ = jax.jit(lambda x: x+1)(jnp.zeros((1,), jnp.float32)); jax.block_until_ready(_)
    from functools import partial
    @partial(jax.jit, out_shardings=out_sh)
    def _init(r): return model.init(r, dummy)["params"]
    p = _init(rng); jax.block_until_ready(p)
    return p

ARCHIVE.write("0.sharding", {"status": "PASS", "n_dev": N_DEV})

[shard] N_DEV=8  mesh=('fsdp',)
  [ARCHIVE] 0.sharding: PASS


In [17]:
# =============================================================================
# 4.1 — enwik8 data
# =============================================================================
import urllib.request, zipfile, shutil
DATA_DIR = f"{WORK_ROOT}/enwik8_data"; os.makedirs(DATA_DIR, exist_ok=True)
ENW = os.path.join(DATA_DIR, "enwik8")

def _ensure():
    if os.path.isfile(ENW) and os.path.getsize(ENW) > 90_000_000: return True
    for root in ("/kaggle/input", "/content"):
        for cand in glob.glob(f"{root}/**/enwik8", recursive=True):
            if os.path.isfile(cand) and os.path.getsize(cand) > 90_000_000:
                shutil.copy(cand, ENW); return True
    try:
        zp = os.path.join(DATA_DIR, "enwik8.zip")
        if not os.path.isfile(zp):
            urllib.request.urlretrieve("https://mattmahoney.net/dc/enwik8.zip", zp)
        with zipfile.ZipFile(zp) as z: z.extract("enwik8", DATA_DIR)
        return True
    except Exception as e:
        print(f"[data] failed: {e}"); return False

if not _ensure(): raise RuntimeError("enwik8 unavailable")
BYTES = np.frombuffer(open(ENW, "rb").read(), dtype=np.uint8)
L = 4096
def make_seqs(bts, L):
    n = len(bts)//L; return bts[:n*L].reshape(n, L)
TRAIN_SEQS = make_seqs(BYTES[:90_000_000], L)
VAL_SEQS = make_seqs(BYTES[90_000_000:95_000_000], L)
print(f"[4.1] train={TRAIN_SEQS.shape} val={VAL_SEQS.shape}")
ARCHIVE.write("4.1", {"status": "PASS", "train_n": int(TRAIN_SEQS.shape[0]),
                       "val_n": int(VAL_SEQS.shape[0]), "L": L})

[4.1] train=(21972, 4096) val=(1220, 4096)
  [ARCHIVE] 4.1: PASS


In [18]:
# =============================================================================
# 4.0 — LM (parametrized)
# =============================================================================
import dataclasses as dc

@dc.dataclass(frozen=True)
class LMConfig:
    vocab_size:int=256; d_model:int=256; n_heads:int=2; d_head:int=128
    n_layers:int=6; mlp_mult:int=2; d_conv:int=4
    g_max:float=5.0; clamp_g:bool=True; decay_init:float=0.05
    @property
    def proj_dim(self): return self.n_heads*self.d_head

class GDN2Block(nn.Module):
    cfg: LMConfig; kernel_fn: object
    @nn.compact
    def __call__(self, x):
        cfg=self.cfg; B,L,_=x.shape; H,D=cfg.n_heads,cfg.d_head; proj_dim=H*D; eps=1e-6
        def sc(name,u):
            dim=u.shape[-1]
            w_=self.param(f"{name}_conv_w", nn.initializers.normal(0.02),(dim,cfg.d_conv))
            bc=self.param(f"{name}_conv_b", nn.initializers.zeros,(dim,))
            rhs=w_.T[:,None,:].astype(u.dtype)
            return jax.lax.conv_general_dilated(u,rhs,window_strides=(1,),
                padding=[(cfg.d_conv-1,0)],feature_group_count=dim,
                dimension_numbers=("NHC","HIO","NHC"))+bc[None,None,:].astype(u.dtype)
        q=jax.nn.silu(sc("q",nn.Dense(proj_dim,use_bias=False,name="q_proj")(x))).reshape(B,L,H,D)
        k=jax.nn.silu(sc("k",nn.Dense(proj_dim,use_bias=False,name="k_proj")(x))).reshape(B,L,H,D)
        v=jax.nn.silu(sc("v",nn.Dense(proj_dim,use_bias=False,name="v_proj")(x))).reshape(B,L,H,D)
        v=jnp.clip(v,-50.0,50.0)
        l2n=lambda t: t*jax.lax.rsqrt(jnp.sum(t*t,-1,keepdims=True)+eps)
        q=l2n(q); k=l2n(k)
        beta=jax.nn.sigmoid(nn.Dense(proj_dim,name="erase_gate")(x)).reshape(B,L,H,D)
        wg=jax.nn.sigmoid(nn.Dense(proj_dim,name="write_gate")(x)).reshape(B,L,H,D)
        bias0=math.log(math.expm1(cfg.decay_init))
        da=self.param("decay_a", lambda k_,s: jnp.full(s,bias0,jnp.float32),(H,)).astype(jnp.float32)
        df=nn.Dense(H,use_bias=True,name="decay_proj")(x).astype(jnp.float32)
        ldh=-jnp.exp(jnp.clip(da,-20,20))[None,None,:]*jax.nn.softplus(df)
        ld=jnp.broadcast_to(ldh[...,None],(B,L,H,D))
        ld=jnp.nan_to_num(ld,nan=0.0,posinf=0.0,neginf=-20.0)
        g=-jnp.minimum(jnp.abs(ld),cfg.g_max) if cfg.clamp_g else -jnp.abs(ld)
        o,_=self.kernel_fn(q,k,v,wg,beta,g); o=o.reshape(B,L,proj_dim)
        og=nn.Dense(proj_dim,use_bias=False,name="out_gate")(x)
        o=nn.RMSNorm(epsilon=1e-6,name="mixer_out_norm")(o)
        x=x+nn.Dense(cfg.d_model,use_bias=False,name="out_proj")(o*jax.nn.silu(og))
        h2=nn.RMSNorm(epsilon=1e-6,name="mlp_norm")(x); d=x.shape[-1]
        m=nn.Dense(d,use_bias=False,name="down_proj")(
            jax.nn.silu(nn.Dense(cfg.mlp_mult*d,use_bias=False,name="gate_proj")(h2))
            *nn.Dense(cfg.mlp_mult*d,use_bias=False,name="up_proj")(h2))
        return x+m

class LM(nn.Module):
    cfg: LMConfig; kernel_fn: object
    @nn.compact
    def __call__(self, ids):
        emb=nn.Embed(num_embeddings=self.cfg.vocab_size,features=self.cfg.d_model,name="embed")
        x=emb(ids)
        for i in range(self.cfg.n_layers):
            x=GDN2Block(cfg=self.cfg,kernel_fn=self.kernel_fn,name=f"block_{i}")(x)
        return emb.attend(nn.RMSNorm(epsilon=1e-6,name="final_norm")(x))
print("[4.0] model defs OK")

[4.0] model defs OK


In [24]:
# =============================================================================
# T-22 PROTOCOL — branches из ОДНОЙ точки, resume из checkpoint
# =============================================================================
CKPT_DIR = f"{WORK_ROOT}/t22_ckpt"; os.makedirs(CKPT_DIR, exist_ok=True)

def make_branch_cfg(branch):
    kw = dict(bt=128, bc=64, score_bs=128, mb=8)
    if branch == "base":      return make_cfg(bs2=128, g_max=1e3, **kw)
    if branch == "bs2_32":    return make_cfg(bs2=32, **kw)
    if branch == "lean":      return make_cfg(bs2=128, g_max=1e3, diag="lean", **kw)
    if branch == "highest":   return make_cfg(bs2=128, g_max=1e3, b3_dot_mode="highest",
                                                solve_dot_mode="highest", **kw)
    raise ValueError(branch)

def run_branch(branch, steps=900, d_model=256, n_heads=2, d_head=128, n_layers=6,
               bsz=8, resume_from=None, ckpt_every=50, lr=3e-4, seed=0):
    print(f"\n{'='*78}\n[T-22] branch={branch}  steps={steps}  resume={resume_from}  seed={seed}\n{'='*78}")
    cfg = make_branch_cfg(branch)
    kfn = make_sharded_blr_kernel(cfg, 1.0/math.sqrt(d_head), n_heads, d_head)
    lmc = LMConfig(vocab_size=256, d_model=d_model, n_heads=n_heads, d_head=d_head,
                   n_layers=n_layers, g_max=cfg.g_max, clamp_g=True, decay_init=0.05)
    model = LM(cfg=lmc, kernel_fn=kfn)
    dummy = jnp.zeros((bsz, L), jnp.int32)

    start_step = 0
    if resume_from and os.path.isfile(resume_from):
        with open(resume_from, "rb") as f:
            state = pickle.load(f)
        params = jax.tree.map(jnp.asarray, state["params"])
        start_step = state["step"]
        print(f"  [resume] loaded {resume_from} at step={start_step}")
    else:
        params = sharded_init(model, jax.random.PRNGKey(seed), dummy)

    sched = optax.warmup_cosine_decay_schedule(0.0, lr, 200, steps, lr*0.1)
    tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(sched, weight_decay=0.1))
    ost = tx.init(params)

    def loss_fn(p, ids):
        lg = model.apply({"params": p}, ids)
        return optax.softmax_cross_entropy_with_integer_labels(lg[:, :-1], ids[:, 1:]).mean()

    @jax.jit
    def step_fn(p, s, ids):
        loss, grads = jax.value_and_grad(loss_fn)(p, ids)
        gn = optax.global_norm(grads)
        upd, s2 = tx.update(grads, s, p)
        return optax.apply_updates(p, upd), s2, loss, gn

    rng_np = np.random.default_rng(seed)
    losses = []; first_nf = None
    t0 = time.perf_counter()
    i = start_step
    for i in range(start_step + 1, steps + 1):
        idx = rng_np.integers(0, len(TRAIN_SEQS), size=bsz)
        ids = jnp.asarray(TRAIN_SEQS[idx], jnp.int32)
        params, ost, loss, gn = step_fn(params, ost, ids)
        lv = float(jax.device_get(loss)); losses.append(lv)
        if not math.isfinite(lv) and first_nf is None:
            first_nf = i; print(f"  [NF] step {i}: loss={lv}")
        if i % ckpt_every == 0:
            with open(os.path.join(CKPT_DIR, f"{branch}_step{i}.pkl"), "wb") as f:
                pickle.dump({"step": i, "params": jax.device_get(params)}, f)
        if i % 100 == 0 or i == start_step + 1:
            print(f"  step={i:>4}/{steps} loss={lv:.4f} gnorm={float(gn):.2f}")
        if first_nf is not None and i - first_nf > 30:
            print(f"  [STOP] non-finite persisted since step {first_nf}"); break
    return dict(branch=branch, steps_run=i, first_nonfinite_step=first_nf,
                final_loss=losses[-1] if losses else None,
                last20=losses[-20:] if losses else [],
                wall_s=round(time.perf_counter()-t0,1))

print("[T-22] run_branch() defined — resumable via CKPT_DIR")

[T-22] run_branch() defined — resumable via CKPT_DIR


In [25]:
# =============================================================================
# 4.2-4.5 — branches из общей точки (все начинают с step=0, seed=0, той же init)
# =============================================================================
results = {}
for branch in ("base", "bs2_32", "lean", "highest"):
    try:
        r = run_branch(branch, steps=900, bsz=8, seed=0)
        results[branch] = r
        if branch == "base":
            st = "INFO" if r["first_nonfinite_step"] is not None else "PASS"
        else:
            st = "FAIL" if r["first_nonfinite_step"] is not None else "PASS"
        ARCHIVE.write(f"4.branch.{branch}", {"status": st, **r})
        print(f"[T-22] {branch}: first_nf={r['first_nonfinite_step']}  final_loss={r['final_loss']}")
    except Exception as e:
        print(f"[T-22] {branch} FAILED: {type(e).__name__}: {str(e)[:200]}")
        ARCHIVE.write(f"4.branch.{branch}", {"status": "ERROR", "reason": str(e)[:300]})

# Интерпретация: T-22 подтверждён, если base НЕ finite, а остальные finite
base_nf = results.get("base", {}).get("first_nonfinite_step")
others_ok = all(results.get(b, {}).get("first_nonfinite_step") is None
                for b in ("bs2_32", "lean", "highest") if b in results)
t22_confirmed = (base_nf is not None) and others_ok
ARCHIVE.write("4.T22_verdict", {"status": "PASS" if t22_confirmed else "INFO",
                                  "base_first_nf": base_nf, "others_finite": others_ok,
                                  "note": "T-22: base spike (bs2=128 unclamped), fixed by bs2=32/lean/highest"})
print(f"\n[T-22 VERDICT] confirmed={t22_confirmed}  base_nf={base_nf}  others_ok={others_ok}")


[T-22] branch=base  steps=900  resume=None  seed=0
  step=   1/900 loss=6.0490 gnorm=5.94
  step= 100/900 loss=2.3328 gnorm=1.78
  step= 200/900 loss=2.0644 gnorm=1.79
  step= 300/900 loss=1.7100 gnorm=1.72
  step= 400/900 loss=1.6151 gnorm=1.43
  step= 500/900 loss=1.5517 gnorm=1.13
  step= 600/900 loss=1.4635 gnorm=1.14
  step= 700/900 loss=1.3528 gnorm=1.26
  step= 800/900 loss=1.4342 gnorm=0.94
  step= 900/900 loss=1.5458 gnorm=1.33
  [ARCHIVE] 4.branch.base: PASS
[T-22] base: first_nf=None  final_loss=1.5458252429962158

[T-22] branch=bs2_32  steps=900  resume=None  seed=0
  step=   1/900 loss=6.0490 gnorm=5.94
  step= 100/900 loss=2.3326 gnorm=1.77
  step= 200/900 loss=2.0646 gnorm=1.82
  step= 300/900 loss=1.7090 gnorm=1.73
  step= 400/900 loss=1.6148 gnorm=1.42
  step= 500/900 loss=1.5515 gnorm=1.13
  step= 600/900 loss=1.4641 gnorm=1.14
  step= 700/900 loss=1.3530 gnorm=1.27
  step= 800/900 loss=1.4342 gnorm=0.94
  step= 900/900 loss=1.5456 gnorm=1.33
  [ARCHIVE] 4.branch.bs2

In [26]:
# =============================================================================
# 4.10 — Scaling: 6M / 25M / 80M / 142M at bs2=32 (safe config)
# =============================================================================
SCALING_MODELS = {
    "6M":   dict(d_model=256,  n_heads=2, n_layers=6),
    "25M":  dict(d_model=512,  n_heads=4, n_layers=6),
    "80M":  dict(d_model=768,  n_heads=6, n_layers=9),
    "142M": dict(d_model=1024, n_heads=8, n_layers=9),
}
print("\n" + "="*78 + "\n4.10 — SCALING\n" + "="*78)
scaling_results = {}

# Для n_params: инициализируем каждую модель и считаем параметры
for tag, kw in SCALING_MODELS.items():
    try:
        cfg = make_branch_cfg("bs2_32")
        kfn = make_sharded_blr_kernel(cfg, 1.0/math.sqrt(128), kw["n_heads"], 128)
        lmc = LMConfig(vocab_size=256, d_model=kw["d_model"], n_heads=kw["n_heads"],
                       d_head=128, n_layers=kw["n_layers"], g_max=cfg.g_max, clamp_g=True)
        model = LM(cfg=lmc, kernel_fn=kfn)
        p_init = sharded_init(model, jax.random.PRNGKey(0), jnp.zeros((8, L), jnp.int32))
        n_params = sum(x.size for x in jax.tree_util.tree_leaves(p_init))
        del p_init; gc.collect()
    except Exception as e:
        print(f"  [{tag}] param-count FAILED: {e}")
        ARCHIVE.write(f"4.10.scale_{tag}", {"status": "ERROR", "reason": str(e)[:300]})
        continue

    try:
        bsz = 8 if tag in ("6M","25M") else 4
        r = run_branch("bs2_32", steps=600, bsz=bsz,
                       d_model=kw["d_model"], n_heads=kw["n_heads"],
                       d_head=128, n_layers=kw["n_layers"], seed=0)
        ppl = math.exp(min(r["final_loss"], 20)) if r["final_loss"] is not None else None
        row = dict(**r, ppl=ppl, n_params=n_params)
        scaling_results[tag] = row
        st = "FAIL" if r["first_nonfinite_step"] is not None else "PASS"
        ARCHIVE.write(f"4.10.scale_{tag}", {"status": st, **row})
        print(f"  [{tag}] n_params={n_params:,}  final_loss={r['final_loss']:.4f}  "
              f"ppl={ppl:.2f}  first_nf={r['first_nonfinite_step']}")
    except Exception as e:
        print(f"  [{tag}] FAILED: {type(e).__name__}: {str(e)[:200]}")
        ARCHIVE.write(f"4.10.scale_{tag}", {"status": "ERROR", "reason": str(e)[:300]})

ARCHIVE.write("4.10.scaling_summary", {
    "status": "PASS",
    "rows": [dict(model=t, **{k: v for k,v in r.items() if k in
                              ("final_loss","ppl","n_params","first_nonfinite_step")})
             for t, r in scaling_results.items()]})


4.10 — SCALING

[T-22] branch=bs2_32  steps=600  resume=None  seed=0
  step=   1/600 loss=6.0490 gnorm=5.94
  step= 100/600 loss=2.3326 gnorm=1.77
  step= 200/600 loss=2.0646 gnorm=1.82
  step= 300/600 loss=1.7104 gnorm=1.49
  step= 400/600 loss=1.6156 gnorm=1.65
  step= 500/600 loss=1.5655 gnorm=1.64
  step= 600/600 loss=1.4947 gnorm=1.19
  [ARCHIVE] 4.10.scale_6M: PASS
  [6M] n_params=5,209,880  final_loss=1.4947  ppl=4.46  first_nf=None

[T-22] branch=bs2_32  steps=600  resume=None  seed=0
  step=   1/600 loss=6.0308 gnorm=8.50
  step= 100/600 loss=2.0129 gnorm=1.94
  step= 200/600 loss=1.8270 gnorm=1.43
  step= 300/600 loss=1.5183 gnorm=1.08
  step= 400/600 loss=1.4445 gnorm=0.91
  step= 500/600 loss=1.3964 gnorm=0.78
  step= 600/600 loss=1.3254 gnorm=0.87
  [ARCHIVE] 4.10.scale_25M: PASS
  [25M] n_params=20,649,520  final_loss=1.3254  ppl=3.76  first_nf=None

[T-22] branch=bs2_32  steps=600  resume=None  seed=0
  step=   1/600 loss=6.0157 gnorm=10.09
  step= 100/600 loss=2.0028 g

In [27]:
# =============================================================================
# 4.11 — Domain trace: half_span / mean|g| vs step (THE T-22 EVIDENCE)
# =============================================================================
def domain_trace(branch, steps=400, bsz=8, stats_every=20, seed=0):
    print(f"\n[4.11] domain_trace branch={branch} steps={steps}")
    cfg = make_branch_cfg(branch)
    kfn_base = make_sharded_blr_kernel(cfg, 1.0/math.sqrt(128), 2, 128)

    captured = []
    def kfn_capture(q, k, v, w, b, g):
        captured.append(g)
        return kfn_base(q, k, v, w, b, g)

    lmc = LMConfig(vocab_size=256, d_model=256, n_heads=2, d_head=128,
                   n_layers=6, g_max=cfg.g_max, clamp_g=True, decay_init=0.05)
    model = LM(cfg=lmc, kernel_fn=kfn_capture)
    params = sharded_init(model, jax.random.PRNGKey(seed), jnp.zeros((bsz, L), jnp.int32))

    sched = optax.warmup_cosine_decay_schedule(0.0, 3e-4, 200, steps, 3e-5)
    tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(sched, weight_decay=0.1))
    ost = tx.init(params)

    def loss_fn(p, ids):
        lg = model.apply({"params": p}, ids)
        return optax.softmax_cross_entropy_with_integer_labels(lg[:, :-1], ids[:, 1:]).mean()

    @jax.jit
    def step_fn(p, s, ids):
        loss, grads = jax.value_and_grad(loss_fn)(p, ids)
        upd, s2 = tx.update(grads, s, p)
        return optax.apply_updates(p, upd), s2, loss

    rng_np = np.random.default_rng(seed)
    rows = []
    bt = 128; nc = L // bt
    for i in range(1, steps + 1):
        idx = rng_np.integers(0, len(TRAIN_SEQS), size=bsz)
        ids = jnp.asarray(TRAIN_SEQS[idx], jnp.int32)
        params, ost, loss = step_fn(params, ost, ids)
        if i % stats_every == 0:
            captured.clear()
            with jax.disable_jit():
                _ = model.apply({"params": params}, ids)
            if captured:
                g_arr = np.asarray(jax.device_get(captured[0]))
                gc_ = np.cumsum(g_arr, axis=1)
                gcb = gc_.reshape(g_arr.shape[0], nc, bt, g_arr.shape[2], g_arr.shape[3])
                spans = 0.5 * (gcb[:, :, 0, :, :] - gcb[:, :, -1, :, :])
                half = float(np.max(np.abs(spans)))
                mean_g = float(np.mean(np.abs(g_arr)))
                rows.append(dict(step=i, loss=float(loss), half_span=half, mean_g=mean_g))
                print(f"  step={i:>4} loss={float(loss):.4f} half_span={half:7.1f} mean|g|={mean_g:.5f}")
    return rows

trace_base = domain_trace("base",   steps=400, seed=0)
trace_bs32 = domain_trace("bs2_32", steps=400, seed=0)

crossed = None
for r in trace_base:
    if r["half_span"] > 88.0:
        crossed = r; break
verdict = "T22_CONFIRMED" if crossed is not None else "T22_NOT_REPRODUCED"

ARCHIVE.write("4.11.domain_trace", {
    "status": "PASS",
    "base": trace_base,
    "bs2_32": trace_bs32,
    "crossed_88_at": crossed,
    "verdict": verdict,
    "note": "half_span crossing 88 on base(bs2=128, unclamped) BEFORE loss blow-up = T-22 evidence",
})
print(f"\n[4.11] T-22 domain evidence: {verdict}  crossed={crossed}")


[4.11] domain_trace branch=base steps=400
  step=  20 loss=5.0395 half_span=    2.3 mean|g|=0.03519
  step=  40 loss=3.6373 half_span=    2.3 mean|g|=0.03507
  step=  60 loss=3.0181 half_span=    2.3 mean|g|=0.03476
  step=  80 loss=2.6230 half_span=    2.3 mean|g|=0.03440
  step= 100 loss=2.3328 half_span=    2.2 mean|g|=0.03388
  step= 120 loss=2.3146 half_span=    2.2 mean|g|=0.03358
  step= 140 loss=2.2347 half_span=    2.2 mean|g|=0.03327
  step= 160 loss=2.1403 half_span=    2.2 mean|g|=0.03298
  step= 180 loss=2.0174 half_span=    2.2 mean|g|=0.03254
  step= 200 loss=2.0644 half_span=    2.1 mean|g|=0.03208
  step= 220 loss=1.9777 half_span=    2.1 mean|g|=0.03149
  step= 240 loss=2.0697 half_span=    2.1 mean|g|=0.03126
  step= 260 loss=1.9368 half_span=    2.2 mean|g|=0.03079
  step= 280 loss=1.8243 half_span=    2.3 mean|g|=0.03076
  step= 300 loss=1.7141 half_span=    2.3 mean|g|=0.03017
  step= 320 loss=1.5275 half_span=    2.1 mean|g|=0.03015
  step= 340 loss=1.7384 half_

In [28]:
# =============================================================================
# 4.99 — SUMMARY (final)
# =============================================================================
print("\n" + "="*78 + "\nNB4 — SUMMARY\n" + "="*78)

print("\n[T-22 branches]")
for b in ("base", "bs2_32", "lean", "highest"):
    r = results.get(b, {})
    print(f"  {b:<10} first_nf={r.get('first_nonfinite_step')}  "
          f"final_loss={r.get('final_loss')}")

print("\n[Scaling]")
for tag, r in scaling_results.items():
    print(f"  {tag:<6} n_params={r.get('n_params',0):>10,}  "
          f"final_loss={r.get('final_loss'):.4f}  ppl={r.get('ppl'):.2f}"
          if r.get('ppl') is not None else
          f"  {tag:<6} n_params={r.get('n_params',0):>10,}  final_loss={r.get('final_loss')}")

print("\n[Domain trace]")
dt = ARCHIVE.data["cells"].get("4.11.domain_trace", {}).get("payload", {})
print(f"  verdict={dt.get('verdict')}  crossed_88={dt.get('crossed_88_at')}")

cells = ARCHIVE.data["cells"]
n_pass = sum(1 for c in cells.values() if c["payload"].get("status") == "PASS")
n_fail = sum(1 for c in cells.values() if c["payload"].get("status") == "FAIL")
n_err  = sum(1 for c in cells.values() if c["payload"].get("status") == "ERROR")
ARCHIVE.write("4.99", {
    "status": "PASS", "n_pass": n_pass, "n_fail": n_fail, "n_error": n_err,
    "t22_verdict": ARCHIVE.data["cells"].get("4.T22_verdict",{}).get("payload",{}).get("status"),
    "domain_trace_verdict": dt.get("verdict"),
})
print(f"\nNB4 — PASS={n_pass}  FAIL={n_fail}  ERROR={n_err}")
print(f"[ARCHIVE] {ARCHIVE.path}")


NB4 — SUMMARY

[T-22 branches]
  base       first_nf=None  final_loss=1.5458252429962158
  bs2_32     first_nf=None  final_loss=1.545578122138977
  lean       first_nf=None  final_loss=1.5458468198776245
  highest    first_nf=None  final_loss=1.5459455251693726

[Scaling]
  6M     n_params= 5,209,880  final_loss=1.4947  ppl=4.46
  25M    n_params=20,649,520  final_loss=1.3254  ppl=3.76
  80M    n_params=69,379,692  final_loss=1.4440  ppl=4.24
  142M   n_params=123,195,536  final_loss=1.4089  ppl=4.09

[Domain trace]
  verdict=T22_NOT_REPRODUCED  crossed_88=None
  [ARCHIVE] 4.99: PASS

NB4 — PASS=13  FAIL=0  ERROR=0
[ARCHIVE] /kaggle/working/attestation/results_nb4.json


In [29]:
# =============================================================================
# 4.11b — T-22 REPRODUCTION: decay_init=1.5 (|g| ~ 2.4, half_span > 88 для bs2=128)
# -----------------------------------------------------------------------------
# FIX от 4.11: half_span считается по bs2-чанкам ветки, а не по bt=128 всегда.
# С decay_init=0.05 (стандарт) |g|~0.035 → half_span~2.2, до 88 не дойдёт.
# С decay_init=1.5  |g|~2.4   → half_span~153 (base, bs2=128) → overflow.
# Для bs2=32 при том же |g|: half_span~38 < 88 → safe.
# =============================================================================
print("\n" + "="*78 + "\n4.11b — T-22 reproduction (decay_init=1.5)\n" + "="*78)

def make_branch_cfg_v2(branch, decay_init):
    kw = dict(bt=128, bc=64, score_bs=128, mb=8)
    if branch == "base":      return make_cfg(bs2=128, g_max=1e3, **kw), decay_init
    if branch == "bs2_32":    return make_cfg(bs2=32,  g_max=1e3, **kw), decay_init
    raise ValueError(branch)

def domain_trace_v2(branch, decay_init, steps=400, bsz=8, stats_every=20, seed=0):
    cfg, di = make_branch_cfg_v2(branch, decay_init)
    bs2_used = cfg.bs2
    print(f"[4.11b] branch={branch}  decay_init={di}  bs2={bs2_used}  steps={steps}")

    kfn_base = make_sharded_blr_kernel(cfg, 1.0/math.sqrt(128), 2, 128)
    captured = []
    def kfn_capture(q, k, v, w, b, g):
        captured.append(g)
        return kfn_base(q, k, v, w, b, g)

    # decay_init передаётся сюда через LMConfig
    lmc = LMConfig(vocab_size=256, d_model=256, n_heads=2, d_head=128, n_layers=6,
                   g_max=cfg.g_max, clamp_g=True, decay_init=di)
    model = LM(cfg=lmc, kernel_fn=kfn_capture)
    params = sharded_init(model, jax.random.PRNGKey(seed), jnp.zeros((bsz, L), jnp.int32))

    sched = optax.warmup_cosine_decay_schedule(0.0, 3e-4, 200, steps, 3e-5)
    tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(sched, weight_decay=0.1))
    ost = tx.init(params)

    def loss_fn(p, ids):
        lg = model.apply({"params": p}, ids)
        return optax.softmax_cross_entropy_with_integer_labels(lg[:, :-1], ids[:, 1:]).mean()
    @jax.jit
    def step_fn(p, s, ids):
        loss, grads = jax.value_and_grad(loss_fn)(p, ids)
        upd, s2 = tx.update(grads, s, p)
        return optax.apply_updates(p, upd), s2, loss

    rng_np = np.random.default_rng(seed)
    rows = []; first_nf = None
    for i in range(1, steps + 1):
        idx = rng_np.integers(0, len(TRAIN_SEQS), size=bsz)
        ids = jnp.asarray(TRAIN_SEQS[idx], jnp.int32)
        params, ost, loss = step_fn(params, ost, ids)
        lv = float(loss)
        if not math.isfinite(lv) and first_nf is None:
            first_nf = i
            print(f"  [NF] step {i}: loss={lv}")
        if i % stats_every == 0:
            captured.clear()
            with jax.disable_jit():
                _ = model.apply({"params": params}, ids)
            if captured:
                g_arr = np.asarray(jax.device_get(captured[0]))  # (B, L, H, D)
                gc_ = np.cumsum(g_arr, axis=1)
                nc_bs2 = L // bs2_used
                gcb = gc_.reshape(g_arr.shape[0], nc_bs2, bs2_used,
                                   g_arr.shape[2], g_arr.shape[3])
                # FIX: half_span по bs2-чанкам этой ветки
                spans = 0.5 * (gcb[:, :, 0, :, :] - gcb[:, :, -1, :, :])
                half = float(np.max(np.abs(spans)))
                mean_g = float(np.mean(np.abs(g_arr)))
                rows.append(dict(step=i, loss=lv, half_span=half, mean_g=mean_g))
                print(f"  step={i:>4} loss={lv:8.4f}  half_span={half:8.1f}  "
                      f"mean|g|={mean_g:.4f}")
        if first_nf is not None and i - first_nf > 30:
            break
    return dict(branch=branch, decay_init=di, bs2=bs2_used,
                rows=rows, first_nf=first_nf)

trace_hi_base = domain_trace_v2("base",   decay_init=1.5, steps=400, seed=0)
trace_hi_bs32 = domain_trace_v2("bs2_32", decay_init=1.5, steps=400, seed=0)

# Evidence: half_span пересекает 88 в base до (или одновременно с) blow-up
crossed_base = next((r for r in trace_hi_base["rows"] if r["half_span"] > 88.0), None)
max_base  = max((r["half_span"] for r in trace_hi_base["rows"]), default=0.0)
max_bs32  = max((r["half_span"] for r in trace_hi_bs32["rows"]), default=0.0)

t22_verdict = "T22_CONFIRMED" if crossed_base else "T22_NOT_REPRODUCED_EVEN_WITH_HIGH_DECAY"
print(f"\n[4.11b] base max half_span = {max_base:.1f}  (crossed 88: {crossed_base is not None})")
print(f"[4.11b] bs2_32 max half_span = {max_bs32:.1f}  (expected < 88)")
print(f"[4.11b] verdict = {t22_verdict}")

ARCHIVE.write("4.11b.t22_high_decay", {
    "status": "PASS",
    "decay_init": 1.5,
    "base":   trace_hi_base,
    "bs2_32": trace_hi_bs32,
    "crossed_88_base": crossed_base,
    "max_half_span_base": max_base,
    "max_half_span_bs2_32": max_bs32,
    "verdict": t22_verdict,
    "note": "decay_init=1.5 → |g|~2.4 → half_span(bs2=128)~153; bs2=32 safe by design",
})


4.11b — T-22 reproduction (decay_init=1.5)
[4.11b] branch=base  decay_init=1.5  bs2=128  steps=400
  step=  20 loss=  4.9790  half_span=   156.1  mean|g|=2.3911
  step=  40 loss=  3.6270  half_span=   155.7  mean|g|=2.3887
  step=  60 loss=  3.1217  half_span=   154.9  mean|g|=2.3751
  step=  80 loss=  2.8255  half_span=   156.0  mean|g|=2.3533
  step= 100 loss=  2.4879  half_span=   151.7  mean|g|=2.3235
  step= 120 loss=  2.4374  half_span=   150.9  mean|g|=2.3000
  step= 140 loss=  2.3475  half_span=   150.2  mean|g|=2.2786
  step= 160 loss=  2.2433  half_span=   149.8  mean|g|=2.2624
  step= 180 loss=  2.0992  half_span=   148.0  mean|g|=2.2464
  step= 200 loss=  2.1425  half_span=   146.9  mean|g|=2.2245
  step= 220 loss=  2.0468  half_span=   146.6  mean|g|=2.1970
  step= 240 loss=  2.1711  half_span=   145.0  mean|g|=2.1800
  step= 260 loss=  2.0374  half_span=   147.0  mean|g|=2.1538
  step= 280 loss=  1.9415  half_span=   150.5  mean|g|=2.1487
  step= 300 loss=  1.7992  half_

In [30]:
# =============================================================================
# 4.10b — SCALING FIX: 2000 steps, bsz=8 (кроме 142M → bsz=4 fallback)
# -----------------------------------------------------------------------------
# Причина предыдущей аномалии (25M < 80M < 142M по loss): 600 шагов — underfit
# для больших моделей. 2000 шагов + bsz=8 (у 80M) должны восстановить монотонность.
# 142M оставлен bsz=4 (OOM-риск); это единственный confound в таблице.
# =============================================================================
print("\n" + "="*78 + "\n4.10b — Scaling fix (2000 steps)\n" + "="*78)

SCALING_V2 = {
    "6M":   dict(d_model=256,  n_heads=2, n_layers=6, bsz=8),
    "25M":  dict(d_model=512,  n_heads=4, n_layers=6, bsz=8),
    "80M":  dict(d_model=768,  n_heads=6, n_layers=9, bsz=8),
    "142M": dict(d_model=1024, n_heads=8, n_layers=9, bsz=4),   # OOM fallback
}

scaling_v2 = {}
for tag, kw in SCALING_V2.items():
    try:
        # param count (без обучения)
        cfg = make_branch_cfg("bs2_32")
        kfn = make_sharded_blr_kernel(cfg, 1.0/math.sqrt(128), kw["n_heads"], 128)
        lmc = LMConfig(vocab_size=256, d_model=kw["d_model"], n_heads=kw["n_heads"],
                       d_head=128, n_layers=kw["n_layers"], g_max=cfg.g_max, clamp_g=True)
        model = LM(cfg=lmc, kernel_fn=kfn)
        p_init = sharded_init(model, jax.random.PRNGKey(0),
                              jnp.zeros((kw["bsz"], L), jnp.int32))
        n_params = sum(x.size for x in jax.tree_util.tree_leaves(p_init))
        del p_init, model; gc.collect(); jax.clear_caches()
    except Exception as e:
        print(f"  [{tag}] param-count FAILED: {e}")
        ARCHIVE.write(f"4.10b.scale_{tag}", {"status": "ERROR", "reason": str(e)[:300]})
        continue

    try:
        r = run_branch("bs2_32", steps=2000, bsz=kw["bsz"],
                       d_model=kw["d_model"], n_heads=kw["n_heads"],
                       d_head=128, n_layers=kw["n_layers"], seed=0)
        ppl = math.exp(min(r["final_loss"], 20)) if r["final_loss"] is not None else None
        row = dict(**r, ppl=ppl, n_params=n_params, bsz=kw["bsz"])
        scaling_v2[tag] = row
        st = "FAIL" if r["first_nonfinite_step"] is not None else "PASS"
        ARCHIVE.write(f"4.10b.scale_{tag}", {"status": st, **row})
        print(f"  [{tag}] n_params={n_params:,}  bsz={kw['bsz']}  "
              f"final_loss={r['final_loss']:.4f}  ppl={ppl:.2f}")
    except Exception as e:
        print(f"  [{tag}] FAILED: {type(e).__name__}: {str(e)[:200]}")
        ARCHIVE.write(f"4.10b.scale_{tag}", {"status": "ERROR", "reason": str(e)[:300]})

# --- Монотонность check ---
if len(scaling_v2) >= 3:
    models_order = ["6M", "25M", "80M", "142M"]
    losses = [scaling_v2.get(m, {}).get("final_loss") for m in models_order]
    finite_losses = [(m, L) for m, L in zip(models_order, losses) if L is not None]
    monotone = all(finite_losses[i][1] >= finite_losses[i+1][1]
                   for i in range(len(finite_losses)-1))
    print(f"\n[4.10b] monotonicity: {monotone}  losses={finite_losses}")
    ARCHIVE.write("4.10b.scaling_monotonicity", {
        "status": "PASS" if monotone else "INFO",
        "monotone": monotone,
        "losses": finite_losses,
        "note": "loss should decrease as model size grows; if not — underfit, "
                "or 142M bsz=4 confound (report explicitly)",
    })


4.10b — Scaling fix (2000 steps)

[T-22] branch=bs2_32  steps=2000  resume=None  seed=0
  step=   1/2000 loss=6.0490 gnorm=5.94
  step= 100/2000 loss=2.3326 gnorm=1.77
  step= 200/2000 loss=2.0646 gnorm=1.82
  step= 300/2000 loss=1.7093 gnorm=1.80
  step= 400/2000 loss=1.6150 gnorm=1.45
  step= 500/2000 loss=1.5512 gnorm=1.25
  step= 600/2000 loss=1.4615 gnorm=1.35
  step= 700/2000 loss=1.3502 gnorm=1.31
  step= 800/2000 loss=1.4202 gnorm=1.07
  step= 900/2000 loss=1.5128 gnorm=1.58
  step=1000/2000 loss=1.3987 gnorm=0.94
  step=1100/2000 loss=1.5045 gnorm=0.94
  step=1200/2000 loss=1.3922 gnorm=1.00
  step=1300/2000 loss=1.3540 gnorm=0.93
  step=1400/2000 loss=1.2756 gnorm=0.85
  step=1500/2000 loss=1.3680 gnorm=1.02
  step=1600/2000 loss=1.2117 gnorm=1.10
  step=1700/2000 loss=1.3811 gnorm=1.02
  step=1800/2000 loss=1.5044 gnorm=0.94
  step=1900/2000 loss=1.4344 gnorm=0.90
  step=2000/2000 loss=1.3002 gnorm=0.93
  [ARCHIVE] 4.10b.scale_6M: PASS
  [6M] n_params=5,209,880  bsz=8  fina

OSError: [Errno 28] No space left on device

In [33]:
# =============================================================================
# t22_ckpt cleanup v2 — final каждого + предпоследний у указанных веток
# =============================================================================
import os, re, glob
from collections import defaultdict

CKPT_DIR = "/content/t22_ckpt" if os.path.isdir("/content/t22_ckpt") else "/kaggle/working/t22_ckpt"
DRY_RUN  = False  # ← поменяй на False чтобы реально удалить

assert os.path.isdir(CKPT_DIR), f"нет директории: {CKPT_DIR}"

# --- политика ---
KEEP_FINAL_OF      = "all"          # финальный чекпоинт каждой ветки
KEEP_PRE_FINAL_FOR = ["bs2_32"]     # ← ветка где сидит 142M: оставить ещё и предпоследний

# --- parse: base_step100.pkl → ("base", 100) ---
def parse(fname):
    m = re.match(r"(.+)_step(\d+)\.pkl$", fname)
    if not m: return None
    return m.group(1), int(m.group(2))

files = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pkl")))
by_branch = defaultdict(list)
for f in files:
    p = parse(os.path.basename(f))
    if p is None: continue
    b, s = p
    by_branch[b].append((s, f))

print(f"[cleanup] dir = {CKPT_DIR}")
print(f"[cleanup] DRY_RUN = {DRY_RUN}")
print(f"[cleanup] keep: final of ALL + pre-final of {KEEP_PRE_FINAL_FOR}")
print(f"[cleanup] total files = {len(files)}\n")

to_keep, to_delete = [], []
for branch in sorted(by_branch.keys()):
    items = sorted(by_branch[branch])   # по возрастанию шага
    n = len(items)

    keep_steps = set()
    if KEEP_FINAL_OF == "all" or branch in KEEP_FINAL_OF:
        keep_steps.add(items[-1][0])                            # финальный
    if branch in KEEP_PRE_FINAL_FOR and n >= 2:
        keep_steps.add(items[-2][0])                            # предпоследний

    branch_kept = [s for s, _ in items if s in keep_steps]
    for s, f in items:
        (to_keep if s in keep_steps else to_delete).append(f)

    print(f"  {branch:<10}  total={n:>3}  keep={len(branch_kept):>2}  "
          f"delete={n - len(branch_kept):>3}  kept_steps={sorted(branch_kept)}")

print(f"\n[cleanup] KEEP   : {len(to_keep)}")
print(f"[cleanup] DELETE : {len(to_delete)}")

if to_delete:
    print("\n  sample deletes (first 15):")
    for f in to_delete[:15]:
        print(f"    - {os.path.basename(f)}")
    if len(to_delete) > 15:
        print(f"    ... +{len(to_delete)-15} more")

if DRY_RUN:
    print(f"\n[cleanup] DRY-RUN — ничего не удалено. Поставь DRY_RUN=False чтобы выполнить.")
else:
    freed = 0
    for f in to_delete:
        try:
            freed += os.path.getsize(f)
            os.remove(f)
        except Exception as e:
            print(f"  [warn] {f}: {e}")
    print(f"\n[cleanup] removed {len(to_delete)} files, freed {freed/1e6:.1f} MB")
    print(f"[cleanup] kept files:")
    for f in sorted(to_keep):
        print(f"    + {os.path.basename(f)}")
    print(f"\n[cleanup] remaining: {len(glob.glob(os.path.join(CKPT_DIR, '*.pkl')))} files")

[cleanup] dir = /kaggle/working/t22_ckpt
[cleanup] DRY_RUN = False
[cleanup] keep: final of ALL + pre-final of ['bs2_32']
[cleanup] total files = 94

  base        total= 18  keep= 1  delete= 17  kept_steps=[900]
  bs2_32      total= 40  keep= 2  delete= 38  kept_steps=[1950, 2000]
  highest     total= 18  keep= 1  delete= 17  kept_steps=[900]
  lean        total= 18  keep= 1  delete= 17  kept_steps=[900]

[cleanup] KEEP   : 5
[cleanup] DELETE : 89

  sample deletes (first 15):
    - base_step50.pkl
    - base_step100.pkl
    - base_step150.pkl
    - base_step200.pkl
    - base_step250.pkl
    - base_step300.pkl
    - base_step350.pkl
    - base_step400.pkl
    - base_step450.pkl
    - base_step500.pkl
    - base_step550.pkl
    - base_step600.pkl
    - base_step650.pkl
    - base_step700.pkl
    - base_step750.pkl
    ... +74 more

[cleanup] removed 89 files, freed 18668.4 MB
[cleanup] kept files:
    + base_step900.pkl
    + bs2_32_step1950.pkl
    + bs2_32_step2000.pkl
    + highest

In [34]:
# =============================================================================
# NB4-diag — 80M gnorm anomaly: isolate shard_map vs architecture
# =============================================================================
%env NB_ID=nb4diag
import os, sys, subprocess, shutil, glob, json, time, math, gc, hashlib
import numpy as np

# --- reuse existing venv (NB4 создал) ---
WORK_ROOT = "/content" if os.path.isdir("/content") else "/kaggle/working"
VENV_DIR  = f"{WORK_ROOT}/venv"
VENV_PY   = f"{VENV_DIR}/bin/python"; VENV_PIP = f"{VENV_DIR}/bin/pip"
_PYTAG    = f"python{sys.version_info.major}.{sys.version_info.minor}"
VENV_SITE = f"{VENV_DIR}/lib/{_PYTAG}/site-packages"
assert os.path.isdir(VENV_SITE), "venv не найден — сначала запусти NB4 Cell 1"
sys.path.insert(0, VENV_SITE)

# --- clone if needed ---
REPO_DIR = f"{WORK_ROOT}/gdn2-test"; PKG_ROOT = os.path.join(REPO_DIR, "pkg")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Akseleu-J/gdn2-test.git",REPO_DIR],
                   capture_output=True, timeout=300)
if PKG_ROOT not in sys.path: sys.path.insert(0, PKG_ROOT)

import jax, jax.numpy as jnp, optax
from flax import linen as nn
from atomic_gdn2 import make_cfg, make_blr_trainable
print(f"[diag] jax={jax.__version__}  backend={jax.default_backend()}  ndev={jax.device_count()}")
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
try: from jax import shard_map
except ImportError: from jax.experimental.shard_map import shard_map

devs = jax.devices(); N_DEV = len(devs)
mesh = Mesh(np.array(devs).reshape(-1), ("fsdp",))
print(f"[diag] N_DEV={N_DEV}")

# =============================================================================
# Model — copy from NB4
# =============================================================================
import dataclasses as dc

@dc.dataclass(frozen=True)
class LMConfig:
    vocab_size:int=256; d_model:int=512; n_heads:int=4; d_head:int=128
    n_layers:int=8; mlp_mult:int=2; d_conv:int=4
    g_max:float=160.0/128; decay_init:float=0.05
    @property
    def proj_dim(self): return self.n_heads * self.d_head

class GDN2Block(nn.Module):
    cfg: LMConfig; kernel_fn: object
    @nn.compact
    def __call__(self, x):
        cfg=self.cfg; B,L,_=x.shape; H,D=cfg.n_heads,cfg.d_head; proj_dim=H*D; eps=1e-6
        def sc(name,u):
            dim=u.shape[-1]
            w_=self.param(f"{name}_conv_w", nn.initializers.normal(0.02),(dim,cfg.d_conv))
            bc=self.param(f"{name}_conv_b", nn.initializers.zeros,(dim,))
            rhs=w_.T[:,None,:].astype(u.dtype)
            return jax.lax.conv_general_dilated(u,rhs,window_strides=(1,),
                padding=[(cfg.d_conv-1,0)],feature_group_count=dim,
                dimension_numbers=("NHC","HIO","NHC"))+bc[None,None,:].astype(u.dtype)
        q=jax.nn.silu(sc("q",nn.Dense(proj_dim,use_bias=False,name="q_proj")(x))).reshape(B,L,H,D)
        k=jax.nn.silu(sc("k",nn.Dense(proj_dim,use_bias=False,name="k_proj")(x))).reshape(B,L,H,D)
        v=jax.nn.silu(sc("v",nn.Dense(proj_dim,use_bias=False,name="v_proj")(x))).reshape(B,L,H,D)
        v=jnp.clip(v,-50.0,50.0)
        l2n=lambda t: t*jax.lax.rsqrt(jnp.sum(t*t,-1,keepdims=True)+eps)
        q=l2n(q); k=l2n(k)
        beta=jax.nn.sigmoid(nn.Dense(proj_dim,name="erase_gate")(x)).reshape(B,L,H,D)
        wg  =jax.nn.sigmoid(nn.Dense(proj_dim,name="write_gate")(x)).reshape(B,L,H,D)
        bias0=math.log(math.expm1(cfg.decay_init))
        da=self.param("decay_a", lambda k_,s: jnp.full(s,bias0,jnp.float32),(H,)).astype(jnp.float32)
        df=nn.Dense(H,use_bias=True,name="decay_proj")(x).astype(jnp.float32)
        ldh=-jnp.exp(jnp.clip(da,-20,20))[None,None,:]*jax.nn.softplus(df)
        ld=jnp.broadcast_to(ldh[...,None],(B,L,H,D))
        ld=jnp.nan_to_num(ld,nan=0.0,posinf=0.0,neginf=-20.0)
        g=-jnp.minimum(jnp.abs(ld),cfg.g_max)
        o,_=self.kernel_fn(q,k,v,wg,beta,g); o=o.reshape(B,L,proj_dim)
        og=nn.Dense(proj_dim,use_bias=False,name="out_gate")(x)
        o=nn.RMSNorm(epsilon=1e-6,name="mixer_out_norm")(o)
        x=x+nn.Dense(cfg.d_model,use_bias=False,name="out_proj")(o*jax.nn.silu(og))
        h2=nn.RMSNorm(epsilon=1e-6,name="mlp_norm")(x); d=x.shape[-1]
        m=nn.Dense(d,use_bias=False,name="down_proj")(
            jax.nn.silu(nn.Dense(cfg.mlp_mult*d,use_bias=False,name="gate_proj")(h2))
            *nn.Dense(cfg.mlp_mult*d,use_bias=False,name="up_proj")(h2))
        return x+m

class LM(nn.Module):
    cfg: LMConfig; kernel_fn: object
    @nn.compact
    def __call__(self, ids):
        emb=nn.Embed(num_embeddings=self.cfg.vocab_size,features=self.cfg.d_model,name="embed")
        x=emb(ids)
        for i in range(self.cfg.n_layers):
            x=GDN2Block(cfg=self.cfg,kernel_fn=self.kernel_fn,name=f"block_{i}")(x)
        return emb.attend(nn.RMSNorm(epsilon=1e-6,name="final_norm")(x))

# =============================================================================
# Kernel factories — controllable shard vs replica
# =============================================================================
CFG_KERNEL = make_cfg(bt=128, bc=64, score_bs=128, bs2=32, mb=8)
try: CFG_KERNEL = CFG_KERNEL.with_(b3_dot_mode="bf16x3", b4_dot_mode="bf16x3", diag="btl")
except Exception: pass
SCALE = 1.0/math.sqrt(128)

def make_kernel(mode, H, D):
    """mode ∈ {'replica', 'shard', 'shard_force_B1'}"""
    fn = make_blr_trainable(CFG_KERNEL, SCALE)
    sw = lambda t: jnp.swapaxes(t, 1, 2)
    def local(q,k,v,w,b,g):
        B = q.shape[0]
        h0 = jnp.zeros((B,H,D,D), jnp.float32)
        o, hf = fn(sw(q),sw(k),sw(v),sw(w),sw(b),sw(g), h0)
        return sw(o), hf
    def call(q,k,v,w,b,g):
        B = q.shape[0]
        if mode == "replica" or N_DEV == 1:
            return local(q,k,v,w,b,g)
        if mode == "shard":
            spec = P("fsdp",None,None,None) if B % N_DEV == 0 else P()
            return shard_map(local, mesh=mesh, in_specs=(spec,)*6,
                             out_specs=(spec,spec), check_vma=False)(q,k,v,w,b,g)
        if mode == "shard_force_B1":
            # обрезаем батч до N_DEV и делаем shard_map, каждый чип видит 1
            assert B % N_DEV == 0
            spec = P("fsdp",None,None,None)
            return shard_map(local, mesh=mesh, in_specs=(spec,)*6,
                             out_specs=(spec,spec), check_vma=False)(q,k,v,w,b,g)
        raise ValueError(mode)
    return call

# =============================================================================
# Diagnostic runner — 80M config, tiny steps, forced config
# =============================================================================
D_MODEL=768; N_HEADS=6; D_HEAD=128; N_LAYERS=9; MLP_MULT=2; D_CONV=4
VOCAB=256; SEQ_LEN=2048   # reduce from 4096 to speed up diagnostic
SCALE_LM = 1.0/math.sqrt(D_HEAD)

def run_diag(label, mode, bsz, steps=100, seed=0, lr=3e-4):
    print(f"\n{'='*78}\n[diag] {label}  mode={mode}  bsz={bsz}  steps={steps}\n{'='*78}")
    kfn = make_kernel(mode, N_HEADS, D_HEAD)
    lmc = LMConfig(vocab_size=VOCAB, d_model=D_MODEL, n_heads=N_HEADS,
                   d_head=D_HEAD, n_layers=N_LAYERS, mlp_mult=MLP_MULT, d_conv=D_CONV,
                   g_max=CFG_KERNEL.g_max, decay_init=0.05)
    model = LM(cfg=lmc, kernel_fn=kfn)
    dummy = jnp.zeros((bsz, SEQ_LEN), jnp.int32)
    params = model.init(jax.random.PRNGKey(seed), dummy)["params"]
    n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
    print(f"  params = {n_params/1e6:.2f}M")

    tx = optax.chain(optax.clip_by_global_norm(1.0),
                     optax.adamw(lr, weight_decay=0.01))
    ost = tx.init(params)

    def loss_fn(p, ids):
        lg = model.apply({"params": p}, ids)
        return optax.softmax_cross_entropy_with_integer_labels(lg[:,:-1], ids[:,1:]).mean()

    @jax.jit
    def step_fn(p, s, ids):
        loss, grads = jax.value_and_grad(loss_fn)(p, ids)
        gn = optax.global_norm(grads)
        upd, s2 = tx.update(grads, s, p)
        return optax.apply_updates(p, upd), s2, loss, gn

    rng = np.random.default_rng(seed)
    rows = []
    t0 = time.perf_counter()
    for i in range(1, steps+1):
        idx = rng.integers(0, 10000, size=bsz)
        ids = jnp.asarray(rng.integers(0, VOCAB, size=(bsz, SEQ_LEN)), jnp.int32)
        params, ost, loss, gn = step_fn(params, ost, ids)
        lv = float(loss); gv = float(gn)
        rows.append({"step": i, "loss": lv, "gnorm": gv})
        if i in (1, 10, 25, 50, 100):
            print(f"  step={i:>4} loss={lv:.4f} gnorm={gv:.2f}")
    print(f"[diag] {label} done in {time.perf_counter()-t0:.0f}s  final loss={rows[-1]['loss']:.4f} gnorm={rows[-1]['gnorm']:.2f}")
    return {"label": label, "mode": mode, "bsz": bsz, "rows": rows}

# --- 4 experiments ---
results = {}
results["A_replica_bs4"] = run_diag("80M replica bsz=4", "replica", 4, steps=100)
results["B_shard_bs8"]   = run_diag("80M shard bsz=8",   "shard",   8, steps=100)
results["C_replica_bs8"] = run_diag("80M replica bsz=8", "replica", 8, steps=100)
results["D_shard_bs16"]  = run_diag("80M shard bsz=16",  "shard",  16, steps=100)

print("\n" + "="*78)
print("SUMMARY — 80M diagnostics")
print("="*78)
print(f"  {'experiment':<20}  {'gnorm@1':>10}  {'gnorm@100':>10}  {'loss@100':>10}")
print("  " + "-"*54)
for k, r in results.items():
    g1 = r["rows"][0]["gnorm"]; g100 = r["rows"][-1]["gnorm"]
    l100 = r["rows"][-1]["loss"]
    print(f"  {k:<20}  {g1:>10.2f}  {g100:>10.2f}  {l100:>10.4f}")

with open(f"{WORK_ROOT}/nb4diag_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n[diag] saved → {WORK_ROOT}/nb4diag_results.json")

env: NB_ID=nb4diag
[diag] jax=0.11.2  backend=tpu  ndev=8
[diag] N_DEV=8

[diag] 80M replica bsz=4  mode=replica  bsz=4  steps=100
  params = 69.38M
  step=   1 loss=6.0367 gnorm=3.92
  step=  10 loss=5.9777 gnorm=3.19
  step=  25 loss=5.9303 gnorm=2.92
  step=  50 loss=5.8800 gnorm=2.43
  step= 100 loss=5.6466 gnorm=0.72
[diag] 80M replica bsz=4 done in 58s  final loss=5.6466 gnorm=0.72

[diag] 80M shard bsz=8  mode=shard  bsz=8  steps=100
  params = 69.38M
  step=   1 loss=6.0373 gnorm=2.79
  step=  10 loss=5.9451 gnorm=2.18
  step=  25 loss=5.8864 gnorm=1.79
  step=  50 loss=5.7525 gnorm=1.01
  step= 100 loss=5.5533 gnorm=0.31
[diag] 80M shard bsz=8 done in 119s  final loss=5.5533 gnorm=0.31

[diag] 80M replica bsz=8  mode=replica  bsz=8  steps=100
  params = 69.38M
  step=   1 loss=6.0374 gnorm=2.79
  step=  10 loss=5.9434 gnorm=2.18
  step=  25 loss=5.8873 gnorm=1.82
  step=  50 loss=5.7588 gnorm=1.03
  step= 100 loss=5.5530 gnorm=0.31
[diag] 80M replica bsz=8 done in 62s  final l

In [ ]:
# =============================================================================
# NB4-diag2 — isolate sharded_init vs plain_init at 80M
# =============================================================================
%env NB_ID=nb4diag2
import os, sys, subprocess, json, time, math, gc
import numpy as np
WORK_ROOT = "/content" if os.path.isdir("/content") else "/kaggle/working"
VENV_SITE = f"{WORK_ROOT}/venv/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages"
REPO_DIR  = f"{WORK_ROOT}/gdn2-test"; PKG_ROOT = os.path.join(REPO_DIR, "pkg")
sys.path.insert(0, VENV_SITE); sys.path.insert(0, PKG_ROOT)

import jax, jax.numpy as jnp, optax
from functools import partial
from flax import linen as nn
from atomic_gdn2 import make_cfg, make_blr_trainable
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
try: from jax import shard_map
except ImportError: from jax.experimental.shard_map import shard_map
from jax.tree_util import tree_flatten_with_path

devs = jax.devices(); N_DEV = len(devs)
mesh = Mesh(np.array(devs).reshape(-1), ("fsdp",))
print(f"[diag2] N_DEV={N_DEV}")

def fsdp_spec(shape, n_dev=N_DEV, axis="fsdp"):
    if n_dev <= 1: return P()
    c = [(s, i) for i, s in enumerate(shape) if s >= n_dev and s % n_dev == 0]
    if not c: return P()
    _, i = max(c)
    spec = [None]*len(shape); spec[i] = axis
    return P(*spec)
def shard_of(x, n_dev=N_DEV):
    return NamedSharding(mesh, fsdp_spec(x.shape, n_dev))

# --- model (copy) ---
import dataclasses as dc
@dc.dataclass(frozen=True)
class LMConfig:
    vocab_size:int=256; d_model:int=768; n_heads:int=6; d_head:int=128
    n_layers:int=9; mlp_mult:int=2; d_conv:int=4
    g_max:float=160.0/128; decay_init:float=0.05
    @property
    def proj_dim(self): return self.n_heads * self.d_head

class GDN2Block(nn.Module):
    cfg: LMConfig; kernel_fn: object
    @nn.compact
    def __call__(self, x):
        cfg=self.cfg; B,L,_=x.shape; H,D=cfg.n_heads,cfg.d_head; proj_dim=H*D; eps=1e-6
        def sc(name,u):
            dim=u.shape[-1]
            w_=self.param(f"{name}_conv_w", nn.initializers.normal(0.02),(dim,cfg.d_conv))
            bc=self.param(f"{name}_conv_b", nn.initializers.zeros,(dim,))
            rhs=w_.T[:,None,:].astype(u.dtype)
            return jax.lax.conv_general_dilated(u,rhs,window_strides=(1,),
                padding=[(cfg.d_conv-1,0)],feature_group_count=dim,
                dimension_numbers=("NHC","HIO","NHC"))+bc[None,None,:].astype(u.dtype)
        q=jax.nn.silu(sc("q",nn.Dense(proj_dim,use_bias=False,name="q_proj")(x))).reshape(B,L,H,D)
        k=jax.nn.silu(sc("k",nn.Dense(proj_dim,use_bias=False,name="k_proj")(x))).reshape(B,L,H,D)
        v=jax.nn.silu(sc("v",nn.Dense(proj_dim,use_bias=False,name="v_proj")(x))).reshape(B,L,H,D)
        v=jnp.clip(v,-50.0,50.0)
        l2n=lambda t: t*jax.lax.rsqrt(jnp.sum(t*t,-1,keepdims=True)+eps)
        q=l2n(q); k=l2n(k)
        beta=jax.nn.sigmoid(nn.Dense(proj_dim,name="erase_gate")(x)).reshape(B,L,H,D)
        wg  =jax.nn.sigmoid(nn.Dense(proj_dim,name="write_gate")(x)).reshape(B,L,H,D)
        bias0=math.log(math.expm1(cfg.decay_init))
        da=self.param("decay_a", lambda k_,s: jnp.full(s,bias0,jnp.float32),(H,)).astype(jnp.float32)
        df=nn.Dense(H,use_bias=True,name="decay_proj")(x).astype(jnp.float32)
        ldh=-jnp.exp(jnp.clip(da,-20,20))[None,None,:]*jax.nn.softplus(df)
        ld=jnp.broadcast_to(ldh[...,None],(B,L,H,D))
        ld=jnp.nan_to_num(ld,nan=0.0,posinf=0.0,neginf=-20.0)
        g=-jnp.minimum(jnp.abs(ld),cfg.g_max)
        o,_=self.kernel_fn(q,k,v,wg,beta,g); o=o.reshape(B,L,proj_dim)
        og=nn.Dense(proj_dim,use_bias=False,name="out_gate")(x)
        o=nn.RMSNorm(epsilon=1e-6,name="mixer_out_norm")(o)
        x=x+nn.Dense(cfg.d_model,use_bias=False,name="out_proj")(o*jax.nn.silu(og))
        h2=nn.RMSNorm(epsilon=1e-6,name="mlp_norm")(x); d=x.shape[-1]
        m=nn.Dense(d,use_bias=False,name="down_proj")(
            jax.nn.silu(nn.Dense(cfg.mlp_mult*d,use_bias=False,name="gate_proj")(h2))
            *nn.Dense(cfg.mlp_mult*d,use_bias=False,name="up_proj")(h2))
        return x+m

class LM(nn.Module):
    cfg: LMConfig; kernel_fn: object
    @nn.compact
    def __call__(self, ids):
        emb=nn.Embed(num_embeddings=self.cfg.vocab_size,features=self.cfg.d_model,name="embed")
        x=emb(ids)
        for i in range(self.cfg.n_layers):
            x=GDN2Block(cfg=self.cfg,kernel_fn=self.kernel_fn,name=f"block_{i}")(x)
        return emb.attend(nn.RMSNorm(epsilon=1e-6,name="final_norm")(x))

CFG_KERNEL = make_cfg(bt=128, bc=64, score_bs=128, bs2=32, mb=8)
try: CFG_KERNEL = CFG_KERNEL.with_(b3_dot_mode="bf16x3", b4_dot_mode="bf16x3", diag="btl")
except Exception: pass
SCALE = 1.0/math.sqrt(128)
H_, D_ = 6, 128
_kfn_raw = make_blr_trainable(CFG_KERNEL, SCALE)
_sw = lambda t: jnp.swapaxes(t, 1, 2)
def kfn_local(q,k,v,w,b,g):
    B=q.shape[0]; h0=jnp.zeros((B,H_,D_,D_),jnp.float32)
    o,hf=_kfn_raw(_sw(q),_sw(k),_sw(v),_sw(w),_sw(b),_sw(g),h0)
    return _sw(o), hf
def kfn(q,k,v,w,b,g):
    B=q.shape[0]; spec=P("fsdp",None,None,None) if B%N_DEV==0 else P()
    return shard_map(kfn_local, mesh=mesh, in_specs=(spec,)*6, out_specs=(spec,spec),
                     check_vma=False)(q,k,v,w,b,g)

BSZ=8; L=2048; VOCAB=256
lmc = LMConfig()
model = LM(cfg=lmc, kernel_fn=kfn)
dummy = jnp.zeros((BSZ, L), jnp.int32)
rng = jax.random.PRNGKey(0)

# --- init 1: plain ---
params_plain = model.init(rng, dummy)["params"]

# --- init 2: sharded_init ---
abstract = jax.eval_shape(lambda r: model.init(r, dummy)["params"], rng)
out_sh = jax.tree.map(shard_of, abstract)
for _ in range(3): gc.collect()
jax.clear_caches()
_ = jax.jit(lambda x: x+1)(jnp.zeros((1,), jnp.float32)); jax.block_until_ready(_)
@partial(jax.jit, out_shardings=out_sh)
def _init_sharded(r): return model.init(r, dummy)["params"]
params_sharded = _init_sharded(rng)

# --- compare params numerics ---
def rel_l2_np(a, b):
    a = np.asarray(jax.device_get(a), np.float64); b = np.asarray(jax.device_get(b), np.float64)
    num = np.linalg.norm((a-b).ravel()); den = np.linalg.norm(b.ravel()) + 1e-12
    return float(num/den)

print("\n[params numerical comparison]")
same = True
for (path_p, leaf_p), (_, leaf_s) in zip(
        tree_flatten_with_path(params_plain)[0], tree_flatten_with_path(params_sharded)[0]):
    r = rel_l2_np(leaf_p, leaf_s)
    if r > 1e-6:
        same = False
        print(f"  DIFF: {'/'.join(str(x.key) for x in path_p if hasattr(x,'key'))}  rel_l2={r:.3e}")
print(f"  params_plain vs params_sharded: identical={same}")

# --- forward + grad on both inits, same data ---
rng_np = np.random.default_rng(0)
ids = jnp.asarray(rng_np.integers(0, VOCAB, size=(BSZ, L)), jnp.int32)

def loss_fn(p):
    lg = model.apply({"params": p}, ids)
    return optax.softmax_cross_entropy_with_integer_labels(lg[:,:-1], ids[:,1:]).mean()

print("\n[gradient comparison @ step 1]")
g_plain   = jax.grad(loss_fn)(params_plain)
g_sharded = jax.grad(loss_fn)(params_sharded)

gn_plain   = float(optax.global_norm(g_plain))
gn_sharded = float(optax.global_norm(g_sharded))
print(f"  optax.global_norm(grads) with plain init   = {gn_plain:.3f}")
print(f"  optax.global_norm(grads) with sharded init = {gn_sharded:.3f}")
print(f"  ratio = {gn_sharded/max(gn_plain,1e-12):.2f}×")

# compare grads numerically per-leaf
print("\n[grads numerical comparison]")
worst = 0.0
for (path_p, leaf_p), (_, leaf_s) in zip(
        tree_flatten_with_path(g_plain)[0], tree_flatten_with_path(g_sharded)[0]):
    r = rel_l2_np(leaf_p, leaf_s)
    worst = max(worst, r)
    if r > 1e-5:
        keypath = "/".join(str(x.key) for x in path_p if hasattr(x, "key"))
        print(f"  DIFF: {keypath}  rel_l2={r:.3e}")
print(f"  worst grad rel_l2 = {worst:.3e}")

# --- also compute manual global norm to check optax's reduction ---
def manual_norm(tree):
    s = 0.0
    for leaf in jax.tree_util.tree_leaves(tree):
        s += float(jnp.sum(leaf.astype(jnp.float32)**2))
    return math.sqrt(s)

print(f"\n  manual norm plain   = {manual_norm(g_plain):.3f}")
print(f"  manual norm sharded = {manual_norm(g_sharded):.3f}")

# --- now check the same with SHARDED PARAMS but with diagnostic = evaluate
# kernel input summary per leaf
print("\n[leaf sharding summary for sharded params]")
n_sharded = 0; n_replicated = 0
for path, leaf in tree_flatten_with_path(params_sharded)[0]:
    try:
        nd = len(leaf.sharding.device_set)
    except Exception:
        nd = 1
    if nd == N_DEV: n_sharded += 1
    else: n_replicated += 1
print(f"  sharded leaves: {n_sharded}  replicated leaves: {n_replicated}")

env: NB_ID=nb4diag2
[diag2] N_DEV=8

[params numerical comparison]
  params_plain vs params_sharded: identical=True

[gradient comparison @ step 1]


In [ ]:
# =============================================================================
# 4.99b — NB4 extra summary
# =============================================================================
print("\n" + "="*78 + "\nNB4 EXTRA SUMMARY\n" + "="*78)

print("\n[T-22 reproduction @ decay_init=1.5]")
d = ARCHIVE.data["cells"].get("4.11b.t22_high_decay", {}).get("payload", {})
print(f"  verdict        : {d.get('verdict')}")
print(f"  base max       : {d.get('max_half_span_base')}")
print(f"  bs2_32 max     : {d.get('max_half_span_bs2_32')}")
print(f"  crossed 88 at  : {d.get('crossed_88_base')}")

print("\n[Scaling @ 2000 steps]")
s = ARCHIVE.data["cells"].get("4.10b.scaling_monotonicity", {}).get("payload", {})
print(f"  monotone       : {s.get('monotone')}")
for m, L in s.get("losses", []):
    print(f"    {m:<6} loss={L:.4f}")

cells = ARCHIVE.data["cells"]
n_pass = sum(1 for c in cells.values() if c["payload"].get("status") == "PASS")
n_fail = sum(1 for c in cells.values() if c["payload"].get("status") == "FAIL")
print(f"\nNB4 total — PASS={n_pass}  FAIL={n_fail}")
print(f"[ARCHIVE] {ARCHIVE.path}")

In [ ]:
# =============================================================================
# 4.11c — T-22 VERDICT: soft bound, quantify the cost
# =============================================================================
rows_base = ARCHIVE.data["cells"]["4.11b.t22_high_decay"]["payload"]["base"]["rows"]
rows_bs32 = ARCHIVE.data["cells"]["4.11b.t22_high_decay"]["payload"]["bs2_32"]["rows"]

# Align по step
common = sorted(set(r["step"] for r in rows_base) & set(r["step"] for r in rows_bs32))
delta_loss = []
for s in common:
    b = next(r for r in rows_base if r["step"] == s)
    t = next(r for r in rows_bs32 if r["step"] == s)
    delta_loss.append({"step": s, "base": b["loss"], "bs2_32": t["loss"],
                       "delta": b["loss"] - t["loss"]})

# Финальная дельта
final = delta_loss[-1] if delta_loss else None
mean_span_base = sum(r["half_span"] for r in rows_base) / len(rows_base)
mean_span_bs32 = sum(r["half_span"] for r in rows_bs32) / len(rows_bs32)
over_bound_factor = mean_span_base / 88.0

verdict = "SOFT_BOUND_CONFIRMED" if (final and 0 < final["delta"] < 0.5) else "INCONCLUSIVE"

print(f"\n[4.11c] T-22 VERDICT:")
print(f"  mean half_span  base   = {mean_span_base:.1f}  ({over_bound_factor:.2f}× over 88)")
print(f"  mean half_span  bs2_32 = {mean_span_bs32:.1f}")
print(f"  final step     = {final['step']}")
print(f"  loss base      = {final['base']:.4f}")
print(f"  loss bs2_32    = {final['bs2_32']:.4f}")
print(f"  Δ loss         = {final['delta']:+.4f}  ({final['delta']/final['bs2_32']*100:+.2f}%)")
print(f"  verdict        = {verdict}")

ARCHIVE.write("4.11c.t22_verdict", {
    "status": "PASS",
    "verdict": verdict,
    "mean_half_span_base": mean_span_base,
    "mean_half_span_bs2_32": mean_span_bs32,
    "over_bound_factor": over_bound_factor,
    "final_step": final["step"] if final else None,
    "final_loss_base": final["base"] if final else None,
    "final_loss_bs2_32": final["bs2_32"] if final else None,
    "final_delta_loss": final["delta"] if final else None,
    "final_delta_pct": (final["delta"]/final["bs2_32"]*100) if final else None,
    "note": "T-22 soft bound: bf16x3 tolerates 1.65× over 88 at cost of ~5% loss",
})

In [ ]:
# =============================================================================
# 4.10b — SCALING FIX: 2000 steps, bsz=8 (кроме 142M → bsz=4 fallback)
# -----------------------------------------------------------------------------
# Причина предыдущей аномалии (25M < 80M < 142M по loss): 600 шагов — underfit
# для больших моделей. 2000 шагов + bsz=8 (у 80M) должны восстановить монотонность.
# 142M оставлен bsz=4 (OOM-риск); это единственный confound в таблице.
# =============================================================================
print("\n" + "="*78 + "\n4.10b — Scaling fix (2000 steps)\n" + "="*78)

SCALING_V2 = {
    
    "80M":  dict(d_model=768,  n_heads=6, n_layers=9, bsz=4),
    
}

scaling_v2 = {}
for tag, kw in SCALING_V2.items():
    try:
        # param count (без обучения)
        cfg = make_branch_cfg("bs2_32")
        kfn = make_sharded_blr_kernel(cfg, 1.0/math.sqrt(128), kw["n_heads"], 128)
        lmc = LMConfig(vocab_size=256, d_model=kw["d_model"], n_heads=kw["n_heads"],
                       d_head=128, n_layers=kw["n_layers"], g_max=cfg.g_max, clamp_g=True)
        model = LM(cfg=lmc, kernel_fn=kfn)
        p_init = sharded_init(model, jax.random.PRNGKey(0),
                              jnp.zeros((kw["bsz"], L), jnp.int32))
        n_params = sum(x.size for x in jax.tree_util.tree_leaves(p_init))
        del p_init, model; gc.collect(); jax.clear_caches()
    except Exception as e:
        print(f"  [{tag}] param-count FAILED: {e}")
        ARCHIVE.write(f"4.10b.scale_{tag}", {"status": "ERROR", "reason": str(e)[:300]})
        continue

    try:
        r = run_branch("bs2_32", steps=2000, bsz=kw["bsz"],
                       d_model=kw["d_model"], n_heads=kw["n_heads"],
                       d_head=128, n_layers=kw["n_layers"], seed=0)
        ppl = math.exp(min(r["final_loss"], 20)) if r["final_loss"] is not None else None
        row = dict(**r, ppl=ppl, n_params=n_params, bsz=kw["bsz"])
        scaling_v2[tag] = row
        st = "FAIL" if r["first_nonfinite_step"] is not None else "PASS"
        ARCHIVE.write(f"4.10b.scale_{tag}", {"status": st, **row})
        print(f"  [{tag}] n_params={n_params:,}  bsz={kw['bsz']}  "
              f"final_loss={r['final_loss']:.4f}  ppl={ppl:.2f}")
    except Exception as e:
        print(f"  [{tag}] FAILED: {type(e).__name__}: {str(e)[:200]}")
        ARCHIVE.write(f"4.10b.scale_{tag}", {"status": "ERROR", "reason": str(e)[:300]})

# --- Монотонность check ---
if len(scaling_v2) >= 3:
    models_order = ["6M", "25M", "80M", "142M"]
    losses = [scaling_v2.get(m, {}).get("final_loss") for m in models_order]
    finite_losses = [(m, L) for m, L in zip(models_order, losses) if L is not None]
    monotone = all(finite_losses[i][1] >= finite_losses[i+1][1]
                   for i in range(len(finite_losses)-1))
    print(f"\n[4.10b] monotonicity: {monotone}  losses={finite_losses}")
    ARCHIVE.write("4.10b.scaling_monotonicity", {
        "status": "PASS" if monotone else "INFO",
        "monotone": monotone,
        "losses": finite_losses,
        "note": "loss should decrease as model size grows; if not — underfit, "
                "or 142M bsz=4 confound (report explicitly)",
    })

In [ ]:
/